# 02 · Draft win predictor
Chance that the Hidden King side wins, from the 12 heroes (plus teammate synergy and cross-team matchup pairs) and the rank gap.
The model lives in `broadway_ml/models/draft.py`; this notebook runs it and shows the evidence. C is tuned on the newest 20% of training games, pairs are kept only if they beat heroes-only on validation, and Platt scaling is folded in only if it helps.
Drafts alone are a weak signal in hero games: an AUC in the mid 0.5s to low 0.6s is normal. What matters for a "54% favoured" widget is calibration.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from broadway_ml.flatten import load_tables, item_catalog
from broadway_ml import features as fe, evaluate as ev, export as ex
T = load_tables('../data/tables')
{k: len(v) for k, v in T.items()}
from broadway_ml.models import draft, live, builds, playstyle, rank
import os; os.makedirs('../models', exist_ok=True)
def plot_rel(table, ax=None, label='model'):
    ax = ax or plt.subplots(figsize=(5, 5))[1]
    ax.plot([0, 1], [0, 1], ls='--', c='grey', lw=1)
    ax.plot(table.predicted, table.observed, marker='o', label=label)
    ax.set(xlabel='predicted win chance', ylabel='observed win rate', xlim=(0, 1), ylim=(0, 1)); ax.legend()
    return ax

In [ ]:
rep = draft.train(T, '../models')
print(rep['variant'], 'C =', rep['C'], '| side advantage (logit):', round(rep['side_advantage_logit'], 4))
rep['comparison']

In [ ]:
plot_rel(rep['reliability'], label='draft model'); rep['reliability']

In [ ]:
pd.Series({**rep['strongest_heroes'], **rep['weakest_heroes']}).sort_values().plot.barh(title='hero effect (logit, + helps its side)');

**Next ideas** once real data is in: train on the current patch only (see 01), add hero x badge interactions (some heroes scale with skill), and compare with a gradient-boosted model to see how much non-linearity is left on the table.